In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# Dataset (MNIST → ImageNet format)

# We must convert:
# 1 channel → 3 channel
# 28×28 → 224×224

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

In [8]:
# Function: prepare pretrained model for MNIST

# We freeze feature extractor and replace classifier.

def prepare_model(model_name):

    if model_name == "resnet":
        model = models.resnet18(weights="DEFAULT")
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, 10)
        classifier = model.fc

    elif model_name == "mobilenet":
        model = models.mobilenet_v3_small(weights="DEFAULT")
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, 10)
        classifier = model.classifier[3]

    elif model_name == "densenet":
        model = models.densenet121(weights="DEFAULT")
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, 10)
        classifier = model.classifier

    # 1) freeze everything
    for param in model.parameters():
        param.requires_grad = False

    # 2) unfreeze classifier ONLY
    for param in classifier.parameters():
        param.requires_grad = True

    return model.to(device)

In [9]:
# Create ensemble models

models_list = [
    prepare_model("resnet"),
    prepare_model("mobilenet"),
    prepare_model("densenet")
]

In [6]:
# Training function

def train_model(model, epochs=1):

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(epochs):
        correct = 0

        for x,y in train_loader:
            x,y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            loss.backward()
            optimizer.step()

            pred = out.argmax(1)
            correct += (pred==y).sum().item()

        acc = correct/len(train_ds)
        print(f"Epoch {epoch+1} Train Acc: {acc:.4f}")

In [10]:
# Train all models

for i,model in enumerate(models_list):
    print(f"\nTraining model {i+1}")
    train_model(model, epochs=1)


Training model 1
Epoch 1 Train Acc: 0.8993

Training model 2
Epoch 1 Train Acc: 0.9263

Training model 3
Epoch 1 Train Acc: 0.8959


In [11]:
# Evaluation: single model accuracy

def test_single(model):
    model.eval()
    correct = 0

    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred==y).sum().item()

    return correct/len(test_ds)

In [12]:
# Evaluation: ensemble accuracy

# We average probabilities.

def test_ensemble(models_list):

    for m in models_list:
        m.eval()

    correct = 0

    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)

            probs = 0
            for m in models_list:
                probs += torch.softmax(m(x),dim=1)

            probs /= len(models_list)
            pred = probs.argmax(1)
            correct += (pred==y).sum().item()

    return correct/len(test_ds)

In [13]:
# Run tests

print("\nSingle model accuracies:")
for i,m in enumerate(models_list):
    print(f"Model {i+1}: {test_single(m):.4f}")

print("\nEnsemble accuracy:")
print(test_ensemble(models_list))


Single model accuracies:
Model 1: 0.9488
Model 2: 0.9544
Model 3: 0.9430

Ensemble accuracy:
0.974
